In [ ]:
import os

os.environ['MODEL'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
os.environ['TOKENIZER'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
os.environ['EVAL'] = 'leaderboard_bbh_boolean_expressions'
os.environ['BATCH'] = str(5)

In [ ]:
!pip install lm-eval lm-eval[math] lm-eval[ifeval] lm-eval[sentencepiece] langdetect

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN=userdata.get('HF')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")

In [ ]:
!kill -9 $(lsof -t -i:8000)
!rm prompts.jsonl
!nohup python server.py $MODEL $TOKENIZER > server.log 2>&1 &

In [ ]:
!rm eval_out*.json
!lm-eval \
    --model local-completions \
    --model_args model=$MODEL,tokenizer=$TOKENIZER,base_url=http://127.0.0.1:8000/v1/completions/true,num_concurrent=1,max_retries=0,tokenized_requests=True,seed=420 \
    --tasks $EVAL \
    --batch_size $BATCH \
    --apply_chat_template \
    --output_path eval_out.json \
    --seed 420 \
    --limit 5